# 02 · Evaluación: de los modelos base a la extracción estructurada

**Tarea 2 – Voces y documentos peruanos.** Se mide la calidad del OCR (CER) y del ASR (WER) contra transcripciones manuales, se extraen campos estructurados con un LLM local, se auditan a mano y se comparan modelos.

**Fuentes y licencias.** Este notebook no agrega datos nuevos: usa los documentos y clips de la Parte 1, todos de **Wikimedia Commons** con licencia libre o en dominio público. El detalle por archivo (autor, licencia y enlace) está en `datos/fuentes.csv` y `datos/atribucion.csv`, y el resumen en la primera celda de `01_extraccion.ipynb`.

**Equipo.** Todo se ejecuta en local, en una MacBook con chip Apple A18 Pro (6 núcleos: 2 de rendimiento y 4 de eficiencia) y **8 GB de RAM**. Por eso se procesa un archivo a la vez, cada modelo se libera antes de cargar el siguiente y los resultados se guardan en disco para retomar si el proceso se corta.

**Modelos que se comparan.** Se fijan aquí, antes de calcular cualquier CER o WER, y no se ajustan después con la muestra de evaluación:

| Modalidad | Línea base (sin ajuste) | Modelo de la Parte 1 | Modelo alternativo (otro proyecto) |
|---|---|---|---|
| Documentos (CER) | PaddleOCR `lang="es"` con sus valores por defecto, CPU | PaddleOCR `PP-OCRv5_mobile_det` + `latin_PP-OCRv5_mobile_rec`, CPU | Tesseract 5 (LSTM), `spa` y `spa_old` en manuscritos, CPU |
| Audio (WER y latencia) | faster-whisper **small**, int8, CPU, VAD | faster-whisper large-v3-turbo, int8, CPU, VAD | NVIDIA Parakeet TDT 0.6B v3 (`parakeet-mlx`, GPU del chip Apple) |

In [6]:
# Para qué: cargar librerías, fijar las carpetas del proyecto y registrar las versiones de los paquetes
import importlib.metadata as md
import platform
from datetime import date
from pathlib import Path

import pandas as pd

TAREA = Path.cwd()
DATOS = TAREA / "datos"
RAW = DATOS / "raw"
REFERENCIAS = RAW / "auditoria"  # muestra y transcripciones de referencia, con nombres reales: no se suben a Git

paquetes = ["paddleocr", "paddlepaddle", "faster-whisper", "ctranslate2", "jiwer", "rapidfuzz", "pandas"]
versiones = pd.DataFrame({"paquete": paquetes, "version": [md.version(p) for p in paquetes]})
print(f"Python {platform.python_version()} · ejecutado el {date.today().isoformat()}")
versiones

Python 3.12.14 · ejecutado el 2026-10-24


,paquete,version
0,paddleocr,3.7.0
1,paddlepaddle,3.3.1
2,faster-whisper,1.2.1
3,ctranslate2,4.8.2
4,jiwer,4.0.0
5,rapidfuzz,3.14.6
6,pandas,2.3.3


## 1. Muestra de evaluación (sin nuevos datos)

Se eligen **10 documentos y 10 clips** de la Parte 1, sin datos nuevos. Son los mismos 20 casos para todos los modelos.

**Documentos:** los tres tipos, con más peso en los manuscritos porque son el caso difícil que hay que analizar.

| Tipo | Documentos | Motivo |
|---|---|---|
| Impreso (3) | `doc_001`, `doc_003`, `doc_007` | Decreto, recomendaciones y comunicado oficial de 2020: caso fácil |
| Foto (3) | `doc_010`, `doc_015`, `doc_016` | Lámina de Guamán Poma en vitrina, cartel y panel en la calle |
| Manuscrito (4) | `doc_018`, `doc_025`, `doc_026`, `doc_028` | Dos páginas de la Relación de Saña (s. XVIII), una partida de bautismo del s. XVIII y el Acta de la Independencia (1821) |

Queda fuera `doc_008` (*El Peruano*, 1844): es muy largo para transcribirlo a mano, y sus columnas mezcladas subirían el CER por el orden de lectura y no por errores de letras.

**Clips:** distintas fuentes y condiciones de grabación, unos 11 minutos en total.

| Condición | Clips |
|---|---|
| Calle con ruido | `clip_002` (El Agustino), `clip_006` (Fiestas Patrias) |
| Multitud o arenga | `clip_004` (hinchas), `clip_009` (Plaza San Martín, 4 voces) |
| Marcha o fiesta, con ambiente | `clip_010` (Marcha del Orgullo 2022), `clip_015` (fiesta de Tayta Pallik) |
| Varias voces | `clip_003` (medio tiempo del partido, 2 voces) |
| Español andino | `clip_013` (Pomabamba) |
| Testimonio en ambiente tranquilo | `clip_016` (San Martín), `clip_018` (proyecto Salas) |

In [7]:
# Para qué: fijar los 10 documentos y los 10 clips de la muestra y comprobar que son casos válidos de la Parte 1
MUESTRA_DOCS = ["doc_001", "doc_003", "doc_007",              # impresos
                "doc_010", "doc_015", "doc_016",              # fotos
                "doc_018", "doc_025", "doc_026", "doc_028"]   # manuscritos
MUESTRA_CLIPS = ["clip_002", "clip_003", "clip_004", "clip_006", "clip_009",
                 "clip_010", "clip_013", "clip_015", "clip_016", "clip_018"]

documentos = pd.read_csv(DATOS / "documentos.csv")
transcripciones = pd.read_csv(DATOS / "transcripciones.csv")

# los 20 casos deben ser archivos válidos de la Parte 1 (no descartados)
assert set(MUESTRA_DOCS) <= set(documentos["id"]), set(MUESTRA_DOCS) - set(documentos["id"])
assert set(MUESTRA_CLIPS) <= set(transcripciones["id"]), set(MUESTRA_CLIPS) - set(transcripciones["id"])

muestra_docs = documentos[documentos["id"].isin(MUESTRA_DOCS)].assign(
    caracteres_ocr=lambda d: d["texto_ocr"].fillna("").str.len())[["id", "tipo", "archivo", "caracteres_ocr"]]
muestra_clips = transcripciones[transcripciones["id"].isin(MUESTRA_CLIPS)].assign(
    palabras_asr=lambda d: d["texto_asr"].fillna("").str.replace(r"\[HABLANTE_\d+\]", "", regex=True)
    .str.split().str.len())[["id", "archivo_audio", "duracion_seg", "n_hablantes", "palabras_asr"]]

print("Documentos por tipo:", muestra_docs["tipo"].value_counts().to_dict())
print(f"Clips: {len(muestra_clips)} · {muestra_clips['duracion_seg'].sum() / 60:.1f} min de audio")
display(muestra_docs)
muestra_clips

Documentos por tipo: {'manuscrito': 4, 'impreso': 3, 'foto': 3}
Clips: 10 · 11.0 min de audio


,id,tipo,archivo,caracteres_ocr
0,doc_001,impreso,datos/raw/documentos/doc_001.jpg,2221
2,doc_003,impreso,datos/raw/documentos/doc_003.jpg,1398
6,doc_007,impreso,datos/raw/documentos/doc_007.jpg,955
9,doc_010,foto,datos/raw/documentos/doc_010.jpg,270
13,doc_015,foto,datos/raw/documentos/doc_015.jpg,826
14,doc_016,foto,datos/raw/documentos/doc_016.jpg,483
16,doc_018,manuscrito,datos/raw/documentos/doc_018.jpg,914
21,doc_025,manuscrito,datos/raw/documentos/doc_025.jpg,416
22,doc_026,manuscrito,datos/raw/documentos/doc_026.jpg,891
24,doc_028,manuscrito,datos/raw/documentos/doc_028.jpg,244


,id,archivo_audio,duracion_seg,n_hablantes,palabras_asr
1,clip_002,datos/raw/audio/clip_002.wav,136.30,2,262
2,clip_003,datos/raw/audio/clip_003.wav,29.93,2,53
3,clip_004,datos/raw/audio/clip_004.wav,20.15,2,29
5,clip_006,datos/raw/audio/clip_006.wav,96.13,2,174
8,clip_009,datos/raw/audio/clip_009.wav,48.25,4,83
9,clip_010,datos/raw/audio/clip_010.wav,54.26,3,159
12,clip_013,datos/raw/audio/clip_013.wav,81.60,1,192
13,clip_015,datos/raw/audio/clip_015.wav,39.24,1,74
14,clip_016,datos/raw/audio/clip_016.wav,43.76,1,119
16,clip_018,datos/raw/audio/clip_018.wav,110.16,1,222


### Transcripción manual

La celda siguiente arma la carpeta `datos/raw/auditoria/` con los 20 casos de la muestra. Por cada caso hay dos archivos:

- **El original:** la imagen (`doc_001.jpg`…) o el audio (`clip_002.wav`…).
- **El borrador** (`doc_001.txt`, `clip_002.txt`…): la salida de la Parte 1, que se **corrige a mano** mirando la imagen o escuchando el audio hasta que diga exactamente lo que está escrito o lo que se dice.
  - En los documentos, es el texto de PaddleOCR (*mobile* latino) en su orden de lectura.
  - En los audios, es el texto de Whisper large-v3-turbo, un segmento por línea con su minuto de inicio (`[1:05]`) para ubicarse al escuchar. Esas marcas de tiempo, como las de hablante, no cuentan en el WER.

Si un caso aún no tiene salida de la Parte 1 (por ejemplo `doc_028` antes de su OCR), el borrador queda vacío y se transcribe desde cero. La celda no pisa un archivo ya corregido. Una copia del borrador sin tocar queda en `datos/raw/auditoria/_borrador/`, y sirve para saber qué casos ya se corrigieron.

**Reglas de corrección:**
- **Documentos:** todo el texto visible, en orden de lectura (columna por columna, de arriba abajo). En los manuscritos se respeta la ortografía original (transcripción paleográfica). Se borra lo que el OCR inventó y se agrega lo que se saltó. Una palabra ilegible se escribe `[ilegible]` y no cuenta en el CER.
- **Audios:** solo las palabras habladas, sin marcas de hablante. Se agregan las palabras omitidas y se borran las que no se dijeron. Lo que no se entiende se escribe `[inaudible]` y no cuenta en el WER.

**Sesgo de este método.** Partir de la salida de un modelo ahorra tiempo, pero puede favorecer a ese modelo: si un error suyo no se nota al revisar, queda en la referencia y no se le cobra. En la comparación de modelos (secciones 4 a 6), la diferencia a favor de PaddleOCR *mobile* y de Whisper turbo (los modelos del borrador) debe leerse con esa cautela. Para reducir el sesgo, cada borrador se revisa línea por línea contra el original y no solo donde «suena raro».

Al volver a ejecutar la celda se ve el avance, sin mostrar el texto.

In [8]:
# Para qué: armar datos/raw/auditoria/ con la imagen o el audio de cada caso y su borrador para corregir a mano;
#           al volver a ejecutarla muestra cuántos borradores ya se corrigieron
import json
import shutil

BORRADOR = REFERENCIAS / "_borrador"
BORRADOR.mkdir(parents=True, exist_ok=True)
OCR_P1 = RAW / "documentos" / "ocr_crudo"
ASR_P1 = RAW / "audio" / "asr"


def borrador(id_):
    # salida de la Parte 1 para corregir a mano ("" si ese caso aún no tiene salida)
    if id_.startswith("doc"):
        txt = OCR_P1 / f"{id_}.txt"
        return txt.read_text(encoding="utf-8") if txt.exists() else ""
    segs = ASR_P1 / f"{id_}_segmentos.json"
    if not segs.exists():
        return ""
    lineas = [f"[{int(s['inicio'] // 60)}:{int(s['inicio'] % 60):02d}] {s['texto'].strip()}"
              for s in json.loads(segs.read_text(encoding="utf-8"))["segmentos"]]
    return "\n".join(lineas) + "\n"


originales = pd.read_csv(DATOS / "fuentes.csv").set_index("id")["archivo"]  # imagen o audio de cada caso
for id_ in MUESTRA_DOCS + MUESTRA_CLIPS:
    origen = TAREA / originales[id_]
    if not (REFERENCIAS / origen.name).exists():
        shutil.copy2(origen, REFERENCIAS / origen.name)
    texto = borrador(id_)
    copia, editable = BORRADOR / f"{id_}.txt", REFERENCIAS / f"{id_}.txt"
    # el borrador se crea (o se completa, si antes no había salida) solo mientras nadie lo haya corregido
    if not editable.exists() or (editable.read_text(encoding="utf-8") == (copia.read_text(encoding="utf-8") if copia.exists() else "")):
        editable.write_text(texto, encoding="utf-8")
        copia.write_text(texto, encoding="utf-8")

avance = pd.DataFrame({"id": MUESTRA_DOCS + MUESTRA_CLIPS})
avance["modalidad"] = avance["id"].str[:3].map({"doc": "documento", "cli": "audio"})
avance["palabras"] = [len((REFERENCIAS / f"{i}.txt").read_text(encoding="utf-8").split()) for i in avance["id"]]
avance["corregido"] = [(REFERENCIAS / f"{i}.txt").read_text(encoding="utf-8") != (BORRADOR / f"{i}.txt").read_text(encoding="utf-8")
                       for i in avance["id"]]
print(f"Carpeta: {REFERENCIAS.relative_to(TAREA)} · corregidos: {avance['corregido'].sum()} de {len(avance)}")
avance

Carpeta: datos/raw/auditoria · corregidos: 5 de 20


,id,modalidad,palabras,corregido
0,doc_001,documento,357,True
1,doc_003,documento,208,True
2,doc_007,documento,149,True
3,doc_010,documento,38,True
4,doc_015,documento,146,True
5,doc_016,documento,78,False
6,doc_018,documento,161,False
7,doc_025,documento,76,False
8,doc_026,documento,135,False
9,doc_028,documento,46,False


### Completar `texto_referencia` en los CSV

Esta celda carga las referencias cuando los 20 borradores están corregidos. Si alguno sigue vacío o sin cambios, se detiene y dice cuál (un borrador que no necesitó ninguna corrección se marca agregando un salto de línea al final).

In [9]:
# Para qué: cargar las 20 transcripciones manuales SIN seudonimizar; son las referencias con las que se mide el CER y el WER
import re

pendientes = [i for i in MUESTRA_DOCS + MUESTRA_CLIPS
              if not (REFERENCIAS / f"{i}.txt").read_text(encoding="utf-8").strip()
              or (REFERENCIAS / f"{i}.txt").read_text(encoding="utf-8") == (BORRADOR / f"{i}.txt").read_text(encoding="utf-8")]
assert not pendientes, f"Faltan por corregir: {pendientes}"

ref_docs = {i: (REFERENCIAS / f"{i}.txt").read_text(encoding="utf-8") for i in MUESTRA_DOCS}
ref_clips = {i: (REFERENCIAS / f"{i}.txt").read_text(encoding="utf-8") for i in MUESTRA_CLIPS}

resumen_ref = pd.DataFrame(
    [{"id": i, "modalidad": "documento", "caracteres": len(t), "palabras": len(t.split()),
      "marcas_ilegible": len(re.findall(r"\[ilegible\]", t, re.I))} for i, t in ref_docs.items()]
    + [{"id": i, "modalidad": "audio", "caracteres": len(t), "palabras": len(t.split()),
        "marcas_ilegible": len(re.findall(r"\[inaudible\]", t, re.I))} for i, t in ref_clips.items()])
print("Longitud de las referencias (sin mostrar el texto):")
resumen_ref

AssertionError: Faltan por corregir: ['doc_016', 'doc_018', 'doc_025', 'doc_026', 'doc_028', 'clip_002', 'clip_003', 'clip_004', 'clip_006', 'clip_009', 'clip_010', 'clip_013', 'clip_015', 'clip_016', 'clip_018']

**Seudonimización de las referencias: solo para publicar en GitHub.** Las referencias se usan de dos formas:

- **Para medir** el CER y el WER se usa la transcripción manual **tal cual, sin seudonimizar** (`ref_docs`, `ref_clips`), igual que la salida de los modelos. Así los dos lados se comparan en las mismas condiciones.
- **Para publicar**, el enunciado pide completar la columna `texto_referencia` de `documentos.csv` y `transcripciones.csv`, que sí se suben a GitHub, y la regla de privacidad prohíbe publicar nombres reales. Por eso las tres celdas siguientes crean una **copia seudonimizada** (`ref_seudo`) que solo se escribe en esos CSV y nunca entra en el cálculo de las métricas.

Las transcripciones manuales pueden traer nombres que el OCR no llegó a leer (sobre todo en la partida de bautismo). Por eso se buscan candidatos otra vez, con las mismas funciones del paso 4 de `01_extraccion.ipynb` y el mismo diccionario privado. En las referencias también se usa la regla de mayúsculas en los documentos, porque una transcripción manual escribe bien los nombres propios.

Los candidatos van a `datos/raw/auditoria/candidatos_referencias.csv`, que no se sube a Git. Aquí solo se cuentan. Si aparece un nombre que no está en el diccionario, se agrega a `diccionario.csv` con su ámbito y se vuelve a ejecutar esta celda.

In [10]:
# SOLO PARA PUBLICAR EN GITHUB: el CER y el WER se calculan con el texto sin seudonimizar (ref_docs, ref_clips)
# Para qué: copiar las funciones de seudonimización de 01_extraccion.ipynb, para aplicarlas a las referencias
# Copiado de 01_extraccion.ipynb (paso 4): mismas funciones y mismo criterio
# ── Extraído de voz.py (sesión 13, Data-Science-Python) ───────────────────────────────
import re
from collections import Counter


def seudonimizar(texto: str, reemplazos: dict[str, str], parecido: int = 80) -> str:
    """Cambia nombres por seudónimos aunque el modelo los haya escrito mal.

    'Ana Vásquez' también atrapa 'Ana Vázquez' (comparación difusa con rapidfuzz).
    [Ejemplo cambiado al copiarlo: el original de voz.py usaba el nombre de una entrevistada.]
    Revisa siempre el resultado: ningún método automático garantiza el anonimato.
    """
    from rapidfuzz import fuzz
    salida = texto
    for nombre, seudonimo in reemplazos.items():
        n = len(nombre.split())
        palabras = re.findall(r"\w+|\W+", salida)
        idx = [i for i, p in enumerate(palabras) if re.match(r"\w", p)]
        cambios = []
        for j in range(len(idx) - n + 1):
            tramo = palabras[idx[j]:idx[j + n - 1] + 1]
            if fuzz.ratio("".join(tramo).lower(), nombre.lower()) >= parecido:
                cambios.append((idx[j], idx[j + n - 1]))
        for a, b in reversed(cambios):
            palabras[a:b + 1] = [seudonimo]
        salida = "".join(palabras)
    return salida


# ── Nuevas para esta tarea ────────────────────────────────────────────────────────────────
# datos personales con forma fija: se reemplazan siempre, sin diccionario
PATRONES = {
    "dni": (r"\b(?:DNI|D\.\s?N\.\s?I\.?)\s*(?:N[°º.o]?\s*)?:?\s*\d{8}\b", "[DNI]"),
    "teléfono": (r"\(0\d{1,2}\)\s?\d{3}[\s-]?\d{4}|\b9\d{2}[\s-]?\d{3}[\s-]?\d{3}\b", "[TELEFONO]"),
    "correo": (r"[\w.+-]+@[\w-]+\.[\w.]+", "[CORREO]"),
}
# nombres después de una pista ("me llamo Ana", "la señora Rosa Quispe", "D. Pedro Pérez")
PISTAS = re.compile(r"(?i:\b(?:me llamo|mi nombre es|soy|señora?|sra?\.|don|doña|d\.|dr\.|dra\.|alcalde|alcaldesa|"
                    r"ministr[oa]|presidente|presidenta|párroco))\s+((?:[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+ ?){1,4})")
# secuencias con mayúscula inicial (Whisper escribe así los nombres propios); incluye lugares: es solo para revisar
MAYUSCULAS = re.compile(r"\b[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+(?:\s+(?:de\s+|del\s+|la\s+)?[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+){1,3}\b")


def aplica(fila, id_):
    return fila.ambito == "*" or id_ in fila.ambito.split("|")


def parecido(fila):
    # umbral de la comparación difusa: 80 como en voz.py, pero 90 para nombres de una sola palabra
    # (con 80, un apellido corto de doc_008 atrapaba la palabra "tres")
    return 90 if len(fila.texto_real.split()) == 1 else 80


def candidatos(id_, corpus, texto, dic):
    # posibles datos personales con su contexto, para la revisión humana
    reglas = [(cat, re.compile(rx)) for cat, (rx, _) in PATRONES.items()] + [("pista de nombre", PISTAS)]
    if corpus == "audio":
        reglas.append(("mayúsculas", MAYUSCULAS))
    salida = []
    for regla, rx in reglas:
        for m in rx.finditer(texto):
            hallado = (m.group(1) if m.groups() else m.group(0)).strip()
            # DNI, teléfonos y correos se reemplazan siempre; los nombres, solo si están en el diccionario
            cubierto = regla in PATRONES or any(
                aplica(f, id_) and seudonimizar(hallado, {f.texto_real: "§"}, parecido(f)) != hallado
                for f in dic.itertuples())
            salida.append({"id": id_, "corpus": corpus, "regla": regla, "texto_detectado": hallado,
                           "contexto": texto[max(0, m.start() - 50):m.end() + 50].replace("\n", " "),
                           "se_reemplaza": cubierto})
    return salida


def seudonimizar_texto(texto, id_, dic):
    # aplica patrones y diccionario; devuelve el texto nuevo y cuántas entidades reemplazó de cada categoría
    conteo = Counter()
    for cat, (rx, reemplazo) in PATRONES.items():
        texto, n = re.subn(rx, reemplazo, texto)
        conteo[cat] += n
    for f in dic.itertuples():  # el diccionario viene ordenado: nombres largos primero
        if aplica(f, id_):
            antes = texto.count(f.seudonimo)
            texto = seudonimizar(texto, {f.texto_real: f.seudonimo}, parecido(f))
            conteo[f.categoria] += texto.count(f.seudonimo) - antes
    return texto, conteo

In [11]:
# SOLO PARA PUBLICAR EN GITHUB: el CER y el WER se calculan con el texto sin seudonimizar (ref_docs, ref_clips)
# Para qué: buscar en las referencias nombres que no están en el diccionario (se guardan en un archivo privado; aquí solo se cuentan)
SEUDO = RAW / "seudonimizacion"
dic = pd.read_csv(SEUDO / "diccionario.csv").fillna({"ambito": "*"})
dic = dic.assign(n_palabras=dic["texto_real"].str.split().str.len()).sort_values("n_palabras", ascending=False)

cand_ref = pd.DataFrame(
    [c for i, t in {**ref_docs, **ref_clips}.items() for c in candidatos(i, "audio", t, dic)],  # "audio" activa la regla de mayúsculas
    columns=["id", "corpus", "regla", "texto_detectado", "contexto", "se_reemplaza"])
cand_ref["corpus"] = cand_ref["id"].str[:3].map({"doc": "documentos", "cli": "audio"})
cand_ref.to_csv(REFERENCIAS / "candidatos_referencias.csv", index=False)
print(f"Diccionario: {len(dic)} entradas")
print("Candidatos en las referencias (se reemplazan / por revisar a mano en candidatos_referencias.csv):")
pd.crosstab([cand_ref["corpus"], cand_ref["regla"]], cand_ref["se_reemplaza"].map({True: "se reemplaza", False: "por revisar"}))

NameError: name 'ref_docs' is not defined

Una vez revisados los candidatos, la referencia seudonimizada se guarda en la columna `texto_referencia` de `documentos.csv` y `transcripciones.csv`. Solo se llena en los 20 casos de la muestra; en los demás queda vacía.

In [12]:
# SOLO PARA PUBLICAR EN GITHUB: el CER y el WER se calculan con el texto sin seudonimizar (ref_docs, ref_clips)
# Para qué: escribir la copia seudonimizada de las referencias en la columna texto_referencia de documentos.csv
#           y transcripciones.csv, que sí se suben a GitHub; ref_docs y ref_clips no cambian
ref_seudo = {i: seudonimizar_texto(t, i, dic)[0] for i, t in {**ref_docs, **ref_clips}.items()}

quedan = sum(seudonimizar(ref_seudo[i], {f.texto_real: "§"}, parecido(f)) != ref_seudo[i]
             for i in ref_seudo for f in dic.itertuples() if aplica(f, i))
assert quedan == 0, f"Quedan {quedan} nombres del diccionario en las referencias"

documentos["texto_referencia"] = documentos["id"].map(ref_seudo)
transcripciones["texto_referencia"] = transcripciones["id"].map(ref_seudo)
documentos.to_csv(DATOS / "documentos.csv", index=False)
transcripciones.to_csv(DATOS / "transcripciones.csv", index=False)
print(f"documentos.csv: {documentos['texto_referencia'].notna().sum()} referencias · "
      f"transcripciones.csv: {transcripciones['texto_referencia'].notna().sum()} referencias")
print(f"Entidades reemplazadas en las referencias: "
      f"{sum(sum(seudonimizar_texto(t, i, dic)[1].values()) for i, t in {**ref_docs, **ref_clips}.items())}")

NameError: name 'ref_docs' is not defined

## 2. Línea base sin ajuste

Se ejecuta la configuración base que pide el enunciado, **PaddleOCR sin personalización adicional** y **faster-whisper small en CPU**, sobre los 10 documentos y los 10 clips de la muestra. Su CER y su WER son la referencia contra la que se comparan los demás modelos.

Cada salida se guarda en `datos/raw/evaluacion/<modelo>/<id>.json` con el texto y el tiempo de proceso; si el proceso se corta, al volver a ejecutar la celda se retoma donde quedó. Un modelo se libera de la memoria antes de cargar el siguiente.

In [13]:
# Para qué: preparar la carpeta de resultados, los hilos de CPU y procesar(), que corre un modelo caso por caso y retoma lo ya hecho
import gc
import json
import os
import time

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

EVAL = RAW / "evaluacion"
HILOS = max(1, min(4, (os.cpu_count() or 4) // 2))  # la mitad de los núcleos: la laptop sigue usable
archivo_doc = documentos.set_index("id")["archivo"]
archivo_clip = transcripciones.set_index("id")["archivo_audio"]
tipo_doc = documentos.set_index("id")["tipo"]
duracion = transcripciones.set_index("id")["duracion_seg"]


def procesar(carpeta: Path, ids: list[str], funcion, modelo: str) -> pd.DataFrame:
    # aplica funcion(id) -> texto a cada caso, guarda texto y segundos, y retoma lo ya hecho
    carpeta.mkdir(parents=True, exist_ok=True)
    filas = []
    for id_ in ids:
        salida = carpeta / f"{id_}.json"
        if salida.exists():
            fila = json.loads(salida.read_text(encoding="utf-8"))
            estado = "ya estaba"
        else:
            t0 = time.perf_counter()
            texto = funcion(id_)
            fila = {"id": id_, "modelo": modelo, "segundos": round(time.perf_counter() - t0, 2), "texto": texto}
            salida.write_text(json.dumps(fila, ensure_ascii=False, indent=1), encoding="utf-8")
            estado = "nuevo"
        filas.append(fila)
        print(f"{id_}  {fila['segundos']:7.1f} s  {len(fila['texto']):6d} caracteres  ({estado})")
    return pd.DataFrame(filas)


print(f"{HILOS} hilos de {os.cpu_count()} núcleos")

3 hilos de 6 núcleos


### PaddleOCR sin personalización adicional

`PaddleOCR(lang="es")` con todos sus valores por defecto (clasificador de orientación, corrección de deformación y detección por defecto). Solo se fija `device="cpu"` y el número de hilos, que no cambian el resultado sino el uso del procesador. El texto se arma en el orden que entrega PaddleOCR, sin el reordenamiento por columnas de la Parte 1. La primera vez descarga sus modelos.

In [ ]:
# Para qué: correr la línea base de documentos (PaddleOCR sin personalización) sobre los 10 documentos y liberar el modelo
from paddleocr import PaddleOCR

MODELO_OCR_BASE = f"PaddleOCR {md.version('paddleocr')} · lang=es, valores por defecto, CPU"
ocr = PaddleOCR(lang="es", device="cpu", cpu_threads=HILOS)  # la primera vez descarga los modelos por defecto


def ocr_base(id_):
    res = ocr.predict(str(TAREA / archivo_doc[id_]))[0]
    return "\n".join(res["rec_texts"])


ocr_paddle_base = procesar(EVAL / "ocr_paddle_base", MUESTRA_DOCS, ocr_base, MODELO_OCR_BASE)
del ocr
gc.collect()
print(MODELO_OCR_BASE)

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Connectivity check to the model hoster has been skipped because `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` is enabled.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/Users/greicyrc/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/Users/greicyrc/Documents/GitHub/Data-Science-Python/.venv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('UVDoc', None, None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `/Users/greicyrc/.paddlex/official_models/UVDoc`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Using official model (PP-LCNet_x1_0_textline_ori), the model files will be automatically downloaded and saved in `/Users/greicyrc/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/greicyrc/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/greicyrc/.paddlex/official_models/PP-OCRv6_medium_rec`.


### faster-whisper small en CPU

`WhisperModel("small", device="cpu", compute_type="int8")` y `transcribe(language="es", vad_filter=True)`, con el resto de valores por defecto. Se guarda solo el texto, sin diarización, porque el WER compara palabras y las marcas de hablante no cuentan.

El **factor de tiempo real** (RTF) es el tiempo de proceso dividido entre la duración del audio: menos de 1 significa que transcribe más rápido de lo que dura el clip.

In [ ]:
# Para qué: correr la línea base de audio (faster-whisper small en CPU) sobre los 10 clips y liberar el modelo
from faster_whisper import WhisperModel

MODELO_ASR_BASE = f"faster-whisper {md.version('faster-whisper')} · Whisper small, int8, CPU {HILOS} hilos, language=es, VAD on"
whisper = WhisperModel("small", device="cpu", compute_type="int8", cpu_threads=HILOS)


def asr_whisper(id_):
    segmentos, _ = whisper.transcribe(str(TAREA / archivo_clip[id_]), language="es", vad_filter=True)
    return " ".join(s.text.strip() for s in segmentos)  # el generador transcribe al recorrerlo


asr_small = procesar(EVAL / "asr_whisper_small", MUESTRA_CLIPS, asr_whisper, MODELO_ASR_BASE)
del whisper
gc.collect()
print(MODELO_ASR_BASE)

### Normalización de las transcripciones

`normalizar()` aplica **el mismo criterio** a la referencia manual y a la salida de cada modelo:

1. Se quitan las marcas que no son texto leído ni palabras habladas: `[HABLANTE_1]`, las marcas de tiempo del borrador (`[1:05]`), `[ilegible]` e `[inaudible]`.
2. Minúsculas y forma Unicode NFC.
3. Se quita la puntuación (los guiones de corte de línea también) y se unifican los espacios y saltos de línea.
4. Se **conservan** las tildes y la ñ, porque en español distinguen palabras.

Las métricas se calculan sobre el texto **sin seudonimizar** de ambos lados, para que el reemplazo de nombres no cambie su longitud; este notebook no imprime nombres reales.

`[ilegible]` e `[inaudible]` marcan tramos que ni la persona pudo leer o entender, así que no se le cobran al modelo.

El CER y el WER se calculan con `jiwer`, por caso y en total. El total suma los errores de todos los casos y los divide entre la longitud total de las referencias, de modo que un caso largo pesa más que uno corto. Ambas métricas pueden pasar de 1 si el modelo agrega mucho texto que no está en la referencia.

In [ ]:
# Para qué: definir la normalización común y medir(), que calcula el CER o el WER por caso y en total
import re
import unicodedata

import jiwer

MARCAS = re.compile(r"\[(?:HABLANTE_\d+|ilegible|inaudible|\d+:\d{2})\]", re.IGNORECASE)  # [1:05] = minuto del borrador


def normalizar(texto: str) -> str:
    # mismo criterio para la referencia y para cada modelo
    texto = MARCAS.sub(" ", texto or "")
    texto = unicodedata.normalize("NFC", texto).lower()
    texto = re.sub(r"-\s*\n\s*", "", texto)              # palabra cortada al final de la línea: "dis-\ntrito"
    texto = re.sub(r"[^\w\s]|_", " ", texto)             # puntuación y símbolos fuera; tildes y ñ se quedan
    return re.sub(r"\s+", " ", texto).strip()


def medir(referencias: dict[str, str], hipotesis: dict[str, str], metrica: str) -> pd.DataFrame:
    # métrica por caso (cer o wer) y una fila "total" con el valor de todo el conjunto
    funcion = jiwer.cer if metrica == "cer" else jiwer.wer
    ids = list(referencias)
    ref = [normalizar(referencias[i]) for i in ids]
    hip = [normalizar(hipotesis.get(i, "")) for i in ids]
    por_caso = [funcion(r, h) if h else 1.0 for r, h in zip(ref, hip)]  # sin salida = todo borrado
    tabla = pd.DataFrame({"id": ids, metrica: por_caso})
    total = funcion(ref, [h or " " for h in hip])
    return pd.concat([tabla, pd.DataFrame({"id": ["total"], metrica: [total]})], ignore_index=True)


# prueba rápida del criterio con frases inventadas
print(normalizar("[HABLANTE_1] ¡Señor Prefecto! El agua corrió por la po-\nblación, ¿no?"))
print(medir({"x": "la casa azul"}, {"x": "La casa, azúl."}, "wer"))

### CER y WER de la línea base

In [ ]:
# Para qué: calcular el CER y el WER de la línea base contra las referencias sin seudonimizar
def textos(tabla):
    return dict(zip(tabla["id"], tabla["texto"]))


cer_base = medir(ref_docs, textos(ocr_paddle_base), "cer")
cer_base["tipo"] = cer_base["id"].map(tipo_doc).fillna("total")
wer_base = medir(ref_clips, textos(asr_small), "wer")
wer_base["rtf"] = wer_base["id"].map(asr_small.set_index("id")["segundos"] / duracion)
wer_base.loc[wer_base["id"] == "total", "rtf"] = asr_small["segundos"].sum() / duracion[MUESTRA_CLIPS].sum()

print(f"Línea base · CER total (PaddleOCR por defecto): {cer_base['cer'].iloc[-1]:.3f}")
print(f"Línea base · WER total (Whisper small): {wer_base['wer'].iloc[-1]:.3f} · RTF {wer_base['rtf'].iloc[-1]:.2f}")
display(cer_base.round(3))
wer_base.round(3)

## 3. Extracción estructurada con LLM y auditoría humana

Se usa **`qwen2.5:3b` en Ollama**, local: los textos no salen de la computadora y no hay costo por consulta. Ocupa unos 2 GB de memoria, así que entra en 8 GB junto al notebook.

El LLM recibe los textos **ya seudonimizados** de `documentos.csv` y `transcripciones.csv` (los 46 del corpus) y debe responder un JSON que cumpla un esquema. Ollama obliga a seguir ese esquema (*structured outputs*), lo que evita JSON roto o campos inventados. Todos los campos aceptan `null` para los datos que no aparecen.

| Modalidad | Campos |
|---|---|
| Documento | `modalidad`, `tema`, `resumen`, `tipo_documento`, `emisor`, `fecha`, `lugar`, `monto` |
| Audio | `modalidad`, `tema`, `resumen`, `numero_hablantes`, `citas_clave`, `lugar` |

Se usa temperatura 0 y semilla fija para que el resultado se pueda repetir. Después de cada respuesta se hace una comprobación automática: cada cita clave debe aparecer literalmente en la transcripción. Esa comprobación no reemplaza la auditoría humana.

In [ ]:
# Para qué: definir el LLM (qwen2.5:3b en Ollama), el esquema JSON de cada modalidad y el prompt; comprueba que Ollama esté listo
import urllib.request
from datetime import datetime

OLLAMA = "http://localhost:11434"
MODELO_LLM = "qwen2.5:3b"
OPCIONES_LLM = {"temperature": 0, "seed": 42, "num_ctx": 8192}
MAX_CARACTERES = 12_000  # el texto más largo del corpus tiene menos; es un tope de seguridad

texto_o_null = {"type": ["string", "null"]}
ESQUEMAS = {
    "documento": {"type": "object", "required": ["modalidad", "tema", "resumen", "tipo_documento", "emisor", "fecha", "lugar", "monto"],
                  "properties": {"modalidad": {"type": "string", "enum": ["documento"]}, "tema": {"type": "string"},
                                 "resumen": {"type": "string"}, "tipo_documento": texto_o_null, "emisor": texto_o_null,
                                 "fecha": texto_o_null, "lugar": texto_o_null, "monto": texto_o_null}},
    "audio": {"type": "object", "required": ["modalidad", "tema", "resumen", "numero_hablantes", "citas_clave", "lugar"],
              "properties": {"modalidad": {"type": "string", "enum": ["audio"]}, "tema": {"type": "string"},
                             "resumen": {"type": "string"}, "numero_hablantes": {"type": ["integer", "null"]},
                             "citas_clave": {"type": ["array", "null"], "items": {"type": "string"}, "maxItems": 3},
                             "lugar": texto_o_null}},
}
INSTRUCCIONES = {
    "documento": """Eres un archivista. Recibes el texto de un documento peruano obtenido por OCR; puede tener errores de lectura.
Extrae los campos del esquema JSON usando SOLO información que aparece en el texto:
- tema: frase corta (máximo 8 palabras) sobre de qué trata.
- resumen: 1 o 2 oraciones en español.
- tipo_documento: decreto, comunicado, cartel, partida de bautismo, relación, etc., si se deduce del texto.
- emisor: institución o cargo que emite el documento, tal como aparece.
- fecha: tal como aparece en el texto (no la conviertas ni la completes).
- lugar: ciudad, provincia o región mencionada como lugar del documento.
- monto: cantidad de dinero mencionada, con su moneda.
Si un dato no aparece, usa null. No inventes. Los textos entre corchetes como [PERSONA_3] son seudónimos: cópialos tal cual.""",
    "audio": """Eres un archivista de historia oral. Recibes la transcripción automática de una entrevista o testimonio en español peruano;
las marcas [HABLANTE_1], [HABLANTE_2]... indican cambios de voz detectados automáticamente y puede haber errores de transcripción.
Extrae los campos del esquema JSON usando SOLO información de la transcripción:
- tema: frase corta (máximo 8 palabras) sobre de qué se habla.
- resumen: 1 o 2 oraciones en español.
- numero_hablantes: cuántas personas distintas hablan, según las marcas [HABLANTE_n]; null si no hay marcas.
- citas_clave: hasta 3 frases copiadas LITERALMENTE de la transcripción, que resuman lo más importante.
- lugar: lugar donde se graba o del que se habla, si se menciona.
Si un dato no aparece, usa null. No inventes. Los textos entre corchetes como [PERSONA_3] son seudónimos: cópialos tal cual.""",
}


def extraer(texto: str, modalidad: str) -> dict:
    cuerpo = {"model": MODELO_LLM, "stream": False, "format": ESQUEMAS[modalidad], "options": OPCIONES_LLM,
              "messages": [{"role": "system", "content": INSTRUCCIONES[modalidad]},
                           {"role": "user", "content": texto[:MAX_CARACTERES]}]}
    pedido = urllib.request.Request(f"{OLLAMA}/api/chat", data=json.dumps(cuerpo).encode(),
                                    headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(pedido, timeout=600) as r:
        return json.loads(json.loads(r.read())["message"]["content"])


# ¿está corriendo Ollama y tiene el modelo?
with urllib.request.urlopen(f"{OLLAMA}/api/tags", timeout=10) as r:
    disponibles = [m["name"] for m in json.loads(r.read())["models"]]
assert MODELO_LLM in disponibles, f"Falta el modelo: ollama pull {MODELO_LLM}"
print(f"Ollama listo · {MODELO_LLM}")

### Guardar `datos/extraccion_llm.csv`

Un texto a la vez, con reanudación (respaldo en `datos/raw/evaluacion/llm/`). Se procesan los 46 textos del corpus (25 documentos y 21 clips) y se escribe `datos/extraccion_llm.csv` con `id`, `modalidad`, `campos_json`, `modelo_llm` y `fecha`.

In [ ]:
# Para qué: extraer los campos JSON de los 46 textos ya seudonimizados y guardar datos/extraccion_llm.csv
corpus_llm = pd.concat([
    documentos[["id", "texto_ocr"]].rename(columns={"texto_ocr": "texto"}).assign(modalidad="documento"),
    transcripciones[["id", "texto_asr"]].rename(columns={"texto_asr": "texto"}).assign(modalidad="audio"),
], ignore_index=True).fillna({"texto": ""})

CARPETA_LLM = EVAL / "llm"
CARPETA_LLM.mkdir(parents=True, exist_ok=True)
filas = []
for t in corpus_llm.itertuples():
    salida = CARPETA_LLM / f"{t.id}.json"
    if salida.exists():
        fila = json.loads(salida.read_text(encoding="utf-8"))
    else:
        t0 = time.perf_counter()
        campos = extraer(t.texto, t.modalidad)
        fila = {"id": t.id, "modalidad": t.modalidad, "campos_json": json.dumps(campos, ensure_ascii=False),
                "modelo_llm": f"{MODELO_LLM} (Ollama, temperatura 0, semilla 42)",
                "fecha": datetime.now().isoformat(timespec="seconds"), "segundos": round(time.perf_counter() - t0, 1)}
        salida.write_text(json.dumps(fila, ensure_ascii=False, indent=1), encoding="utf-8")
    filas.append(fila)
    print(f"{t.id:9} {fila['segundos']:6.1f} s")

extraccion = pd.DataFrame(filas)
extraccion[["id", "modalidad", "campos_json", "modelo_llm", "fecha"]].to_csv(DATOS / "extraccion_llm.csv", index=False)
print(f"\nGuardado datos/extraccion_llm.csv: {extraccion.shape} · {extraccion['segundos'].sum() / 60:.1f} min")

In [ ]:
# Para qué: revisar automáticamente las extracciones (campos en null y citas que no aparecen en la transcripción)
# comprobaciones automáticas: campos nulos y citas que no aparecen literalmente en la transcripción
campos = pd.DataFrame([json.loads(c) for c in extraccion["campos_json"]]).assign(id=extraccion["id"])
texto_de = corpus_llm.set_index("id")["texto"]


def citas_inventadas(fila):
    citas = fila["citas_clave"] if isinstance(fila.get("citas_clave"), list) else []
    return sum(normalizar(c) not in normalizar(texto_de[fila["id"]]) for c in citas)


aud = campos[campos["modalidad"] == "audio"]
print("Campos con null por modalidad:")
display(campos.drop(columns="id").groupby("modalidad").agg(lambda s: s.isna().sum()))
print(f"Citas clave que no aparecen literalmente: {sum(citas_inventadas(f) for _, f in aud.iterrows())} "
      f"de {aud['citas_clave'].dropna().str.len().sum()}")
extraccion[["id", "modalidad", "campos_json"]].head(4)

### Auditoría humana

Se eligen **20 extracciones al azar** (10 documentos y 10 audios, semilla 42) y se revisa cada campo contra el texto y contra el archivo original. La celda crea la plantilla `datos/auditoria_llm.csv`, con una fila por campo. Se llena a mano la columna `correcto` (1 o 0) y, si hace falta, `comentario`. Los textos ya están seudonimizados, así que la plantilla se puede publicar.

**Criterios de corrección por campo:**

| Campo | Se marca correcto (1) si… |
|---|---|
| `tema`, `resumen` | Describe lo que dice el texto, sin datos que no estén en él. Un resumen incompleto pero fiel cuenta como correcto. |
| `tipo_documento`, `emisor`, `lugar` | Coincide con el documento o el audio, o es `null` y el dato de verdad no aparece. |
| `fecha`, `monto` | Es la fecha o el monto que aparece, sin inventar partes; o `null` si no aparece. |
| `numero_hablantes` | Coincide con las voces que se oyen en el audio (no solo con las marcas del diarizador). |
| `citas_clave` | Las citas aparecen en la transcripción y son relevantes. |
| Cualquier campo | Es incorrecto (0) si inventa un dato, deja `null` un dato que sí aparece, o confunde el dato (por ejemplo, el destinatario como emisor). |

Una extracción es **completamente correcta** si todos sus campos valen 1.

In [ ]:
# Para qué: elegir al azar las 20 extracciones a auditar y crear la plantilla datos/auditoria_llm.csv para llenarla a mano
AUDITORIA = DATOS / "auditoria_llm.csv"
auditados = pd.concat([extraccion[extraccion["modalidad"] == m].sample(10, random_state=42)
                       for m in ["documento", "audio"]])["id"].tolist()

if not AUDITORIA.exists():  # no pisar una auditoría ya empezada
    plantilla = [{"id": f.id, "modalidad": f.modalidad, "campo": k,
                  "valor_llm": json.dumps(v, ensure_ascii=False) if isinstance(v, list) else v,
                  "correcto": None, "comentario": None}
                 for f in extraccion[extraccion["id"].isin(auditados)].itertuples()
                 for k, v in json.loads(f.campos_json).items() if k != "modalidad"]
    pd.DataFrame(plantilla).to_csv(AUDITORIA, index=False)
print(f"Extracciones a auditar ({len(auditados)}): {', '.join(sorted(auditados))}")
print(f"Plantilla: {AUDITORIA.relative_to(TAREA)}")

In [ ]:
# Para qué: leer la auditoría ya llenada y reportar las extracciones completamente correctas sobre 20 y los errores
# resultado de la auditoría (ejecutar después de llenar la columna "correcto")
aud = pd.read_csv(AUDITORIA)
assert aud["correcto"].notna().all(), f"Faltan {aud['correcto'].isna().sum()} campos por auditar"

por_extraccion = aud.groupby(["id", "modalidad"])["correcto"].min().rename("completamente_correcta").reset_index()
n_ok = int(por_extraccion["completamente_correcta"].sum())
print(f"Extracciones completamente correctas: {n_ok}/20 = {n_ok / 20:.0%}")
print(por_extraccion.groupby("modalidad")["completamente_correcta"].agg(["sum", "count"]).to_string())
print("\nAcierto por campo:")
display(aud.groupby(["modalidad", "campo"])["correcto"].agg(["mean", "count"]).round(2))
print("Ejemplos de errores:")
aud[aud["correcto"] == 0][["id", "campo", "valor_llm", "comentario"]]

## 4. Mejora y comparación de modelos

Los mismos 10 documentos y 10 clips de la muestra, con el mismo criterio de normalización. En cada modalidad se comparan tres modelos: la línea base, el modelo usado en la Parte 1 y un modelo alternativo de otro proyecto.

### Documentos: PaddleOCR frente a Tesseract

- **PaddleOCR de la Parte 1** (`PP-OCRv5_mobile_det` + `latin_PP-OCRv5_mobile_rec`, sin corrección de deformación, detección a ≤1920 px y lectura por columnas): se reutiliza el texto crudo y el tiempo ya guardados en `datos/raw/documentos/ocr_crudo/`, sin volver a ejecutarlo.
- **Tesseract 5** (motor LSTM, proyecto de código abierto mantenido originalmente por HP y luego por Google) con el modelo `spa`, y `spa_old` (español antiguo, con ſ y grafías de los siglos XVI–XVIII) en los manuscritos. Es el OCR libre más usado, corre en CPU con muy poca memoria y sirve de contraste a una red de detección y reconocimiento como PaddleOCR. Se llama desde la línea de comandos, sin paquetes extra. Ninguno de los dos tiene un modelo de letra manuscrita.

In [ ]:
# Para qué: correr Tesseract sobre los 10 documentos y recuperar el texto y el tiempo de PaddleOCR de la Parte 1
import subprocess

version_tesseract = subprocess.run(["tesseract", "--version"], capture_output=True, text=True).stdout.split("\n")[0]
MODELO_TESSERACT = f"{version_tesseract} · LSTM, spa (manuscritos: spa_old), psm 3"


def ocr_tesseract(id_):
    idioma = "spa_old" if tipo_doc[id_] == "manuscrito" else "spa"
    salida = subprocess.run(["tesseract", str(TAREA / archivo_doc[id_]), "stdout", "-l", idioma, "--psm", "3"],
                            capture_output=True, text=True, check=True, env={**os.environ, "OMP_THREAD_LIMIT": str(HILOS)})
    return salida.stdout


ocr_tess = procesar(EVAL / "ocr_tesseract", MUESTRA_DOCS, ocr_tesseract, MODELO_TESSERACT)

# configuración de la Parte 1: texto y tiempo ya guardados
CRUDO = RAW / "documentos" / "ocr_crudo"
p1 = [json.loads((CRUDO / f"{i}.json").read_text(encoding="utf-8")) for i in MUESTRA_DOCS]
ocr_paddle_p1 = pd.DataFrame({"id": MUESTRA_DOCS, "segundos": [c["segundos"] for c in p1],
                              "texto": ["\n".join(l["texto"] for l in c["lineas"]) for c in p1]})
MODELO_OCR_P1 = p1[0]["modelo_ocr"]
print(MODELO_TESSERACT)

In [ ]:
# Para qué: calcular el CER de los tres modelos de OCR en los mismos 10 documentos
def con_tiempos(metricas: pd.DataFrame, tabla: pd.DataFrame) -> pd.DataFrame:
    # agrega los segundos de proceso por caso y su suma en la fila "total"
    seg = metricas["id"].map(tabla.set_index("id")["segundos"])
    seg[metricas["id"] == "total"] = tabla["segundos"].sum()
    return metricas.assign(segundos=seg)


modelos_ocr = {"PaddleOCR por defecto (línea base)": (ocr_paddle_base, MODELO_OCR_BASE),
               "PaddleOCR mobile latino (Parte 1)": (ocr_paddle_p1, MODELO_OCR_P1),
               "Tesseract spa / spa_old": (ocr_tess, MODELO_TESSERACT)}
cer_modelos = pd.concat([con_tiempos(medir(ref_docs, textos(tabla), "cer"), tabla).assign(modelo=nombre)
                         for nombre, (tabla, _) in modelos_ocr.items()], ignore_index=True)
cer_modelos["tipo"] = cer_modelos["id"].map(tipo_doc).fillna("total")

tabla_cer = cer_modelos.pivot(index="id", columns="modelo", values="cer")[list(modelos_ocr)]
tabla_cer.insert(0, "tipo", tipo_doc.reindex(tabla_cer.index).fillna("total"))
tabla_cer.loc[MUESTRA_DOCS + ["total"]].round(3)

### Audio: faster-whisper frente a Parakeet

- **faster-whisper large-v3-turbo** (Parte 1, int8 en CPU, VAD, beam 5): se reutiliza el texto y el tiempo de transcripción ya guardados en `datos/raw/audio/asr/`. El tiempo es solo el de la transcripción, sin la diarización.
- **NVIDIA Parakeet TDT 0.6B v3**: un ASR de otro proyecto (NVIDIA NeMo), con codificador FastConformer y decodificador TDT, entrenado en 25 idiomas europeos, entre ellos el español. Se ejecuta con `parakeet-mlx`, que usa la GPU del chip Apple, y ocupa unos 1,2 GB. Ojo al comparar latencias: Whisper corre en la CPU y Parakeet en la GPU.

La **latencia** se mide con el **factor de tiempo real** (RTF): el tiempo de procesamiento dividido entre la duración del audio. Un RTF menor que 1 significa que el modelo transcribe más rápido de lo que dura el clip.

Instalación de Parakeet (una vez, en el entorno del curso; ojo, un `uv sync` posterior lo desinstala, como a plotly):

```bash
uv pip install --python ~/Documents/GitHub/Data-Science-Python/.venv/bin/python parakeet-mlx
```

In [ ]:
# Para qué: recuperar el texto y el tiempo de faster-whisper large-v3-turbo de la Parte 1, sin volver a ejecutarlo
# Whisper large-v3-turbo de la Parte 1: texto sin marcas de hablante y tiempo de transcripción
ASR_P1 = RAW / "audio" / "asr"
p1 = [json.loads((ASR_P1 / f"{i}_segmentos.json").read_text(encoding="utf-8")) for i in MUESTRA_CLIPS]
asr_turbo = pd.DataFrame({"id": MUESTRA_CLIPS, "segundos": [c["segundos_proceso"] for c in p1],
                          "texto": [" ".join(s["texto"].strip() for s in c["segmentos"]) for c in p1]})
MODELO_ASR_TURBO = p1[0]["modelo_asr"].split(" · diarización")[0]
print(MODELO_ASR_TURBO)

In [ ]:
# Para qué: correr Parakeet sobre los 10 clips y liberar el modelo
from parakeet_mlx import from_pretrained

MODELO_HF_PARAKEET = "mlx-community/parakeet-tdt-0.6b-v3"
MODELO_PARAKEET = f"parakeet-mlx {md.version('parakeet-mlx')} · {MODELO_HF_PARAKEET}, GPU Apple (MLX), trozos de 120 s"
parakeet = from_pretrained(MODELO_HF_PARAKEET)  # la primera vez descarga ~1,2 GB


def asr_parakeet(id_):
    # los clips de más de 2 min se procesan en trozos de 120 s con 15 s de solape
    return parakeet.transcribe(str(TAREA / archivo_clip[id_]), chunk_duration=120.0, overlap_duration=15.0).text


asr_parak = procesar(EVAL / "asr_parakeet", MUESTRA_CLIPS, asr_parakeet, MODELO_PARAKEET)
del parakeet
gc.collect()
print(MODELO_PARAKEET)

In [ ]:
# Para qué: calcular el WER y la latencia (segundos y RTF) de los tres modelos de ASR en los mismos 10 clips
modelos_asr = {"faster-whisper small (línea base)": (asr_small, MODELO_ASR_BASE),
               "faster-whisper large-v3-turbo (Parte 1)": (asr_turbo, MODELO_ASR_TURBO),
               "Parakeet TDT 0.6B v3": (asr_parak, MODELO_PARAKEET)}
wer_modelos = pd.concat([con_tiempos(medir(ref_clips, textos(tabla), "wer"), tabla).assign(modelo=nombre)
                         for nombre, (tabla, _) in modelos_asr.items()], ignore_index=True)
dur = wer_modelos["id"].map(duracion)
dur[wer_modelos["id"] == "total"] = duracion[MUESTRA_CLIPS].sum()
wer_modelos["rtf"] = wer_modelos["segundos"] / dur

totales = wer_modelos[wer_modelos["id"] == "total"].set_index("modelo").loc[list(modelos_asr)]
print("WER y latencia en los 10 clips "
      f"({duracion[MUESTRA_CLIPS].sum() / 60:.1f} min de audio):")
display(totales[["wer", "segundos", "rtf"]].round(3))
wer_modelos.pivot(index="id", columns="modelo", values="wer")[list(modelos_asr)].loc[MUESTRA_CLIPS + ["total"]].round(3)

## 5. Casos difíciles

**Manuscritos**: se compara el CER de cada tipo de documento (impreso, foto, manuscrito) con cada modelo. El valor de cada tipo suma errores y longitudes de sus documentos.

**Audio con ruido o música**: los clips de la muestra se agrupan por su condición de grabación (ver la sección 1) y se compara el WER de cada grupo con cada modelo.

El bonus opcional de quechua de Huaylas con MMS no se realiza: el corpus no tiene audio en quechua.

In [ ]:
# Para qué: caso difícil de documentos; CER por tipo (impreso, foto, manuscrito) con cada modelo, en tabla y gráfico
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+png"  # interactivo en VS Code y PNG en GitHub
# gris = línea base, azul = Parte 1, naranja = modelo de otro proyecto
COLORES_MODELO = dict(zip(list(modelos_ocr) + list(modelos_asr), ["#a3a29b", "#2a78d6", "#eb6834"] * 2))

cer_tipo = pd.DataFrame([{"modelo": nombre, "tipo": t,
                          "CER": jiwer.cer([normalizar(ref_docs[i]) for i in ids],
                                           [normalizar(textos(tabla).get(i, "")) or " " for i in ids])}
                         for nombre, (tabla, _) in modelos_ocr.items()
                         for t, ids in pd.Series(MUESTRA_DOCS).groupby(tipo_doc[MUESTRA_DOCS].values)])
display(cer_tipo.pivot(index="tipo", columns="modelo", values="CER").loc[["impreso", "foto", "manuscrito"], list(modelos_ocr)].round(3))

fig = px.bar(cer_tipo, x="tipo", y="CER", color="modelo", barmode="group", color_discrete_map=COLORES_MODELO,
             category_orders={"tipo": ["impreso", "foto", "manuscrito"], "modelo": list(modelos_ocr)},
             title="CER por tipo de documento (menor es mejor)")
fig.update_layout(template="plotly_white", legend_title_text="", yaxis_title="CER", xaxis_title="")
fig.show()

In [ ]:
# Para qué: caso difícil de audio; WER por condición de grabación (ruido, multitud, música...) con cada modelo
CONDICION = {"clip_002": "calle con ruido", "clip_006": "calle con ruido",
             "clip_004": "multitud o arenga", "clip_009": "multitud o arenga",
             "clip_010": "marcha o fiesta", "clip_015": "marcha o fiesta",
             "clip_003": "varias voces", "clip_013": "español andino",
             "clip_016": "ambiente tranquilo", "clip_018": "ambiente tranquilo"}
ORDEN_CONDICION = ["calle con ruido", "multitud o arenga", "marcha o fiesta", "varias voces", "español andino", "ambiente tranquilo"]

wer_cond = pd.DataFrame([{"modelo": nombre, "condicion": c,
                          "WER": jiwer.wer([normalizar(ref_clips[i]) for i in ids],
                                           [normalizar(textos(tabla).get(i, "")) or " " for i in ids])}
                         for nombre, (tabla, _) in modelos_asr.items()
                         for c, ids in pd.Series(list(CONDICION)).groupby(list(CONDICION.values()))])
display(wer_cond.pivot(index="condicion", columns="modelo", values="WER").loc[ORDEN_CONDICION, list(modelos_asr)].round(3))

fig = px.bar(wer_cond, x="condicion", y="WER", color="modelo", barmode="group", color_discrete_map=COLORES_MODELO,
             category_orders={"condicion": ORDEN_CONDICION, "modelo": list(modelos_asr)},
             title="WER por condición de grabación (menor es mejor)")
fig.update_layout(template="plotly_white", legend_title_text="", yaxis_title="WER", xaxis_title="")
fig.show()

**Observaciones de los casos difíciles**

*(Se escribe después de ver los resultados: manuscritos frente a impresos y fotos, clips con ruido o música frente a los de ambiente tranquilo, y la causa de cada diferencia.)*

## 6. Comparativa y decisión

Tabla de todos los modelos: error total en la muestra, mediana por caso, tiempo de proceso y, en audio, RTF. Se guarda en `datos/comparativa_modelos.csv`, y las métricas por caso en `datos/metricas_por_caso.csv`.

In [ ]:
# Para qué: armar la tabla comparativa de todos los modelos y guardar comparativa_modelos.csv y metricas_por_caso.csv
ROL = dict(zip(list(modelos_ocr) + list(modelos_asr), ["línea base", "Parte 1", "otro proyecto"] * 2))
por_caso = pd.concat([cer_modelos.rename(columns={"cer": "error"}).assign(modalidad="documento", metrica="CER"),
                      wer_modelos.rename(columns={"wer": "error"}).assign(modalidad="audio", metrica="WER")],
                     ignore_index=True)
por_caso["rol"] = por_caso["modelo"].map(ROL)
casos = por_caso[por_caso["id"] != "total"]

comparativa = (por_caso[por_caso["id"] == "total"][["modalidad", "rol", "modelo", "metrica", "error", "segundos", "rtf"]]
               .merge(casos.groupby("modelo")["error"].median().rename("mediana_por_caso"), on="modelo"))
comparativa["seg_por_caso"] = comparativa["segundos"] / 10
comparativa["equipo"] = comparativa["modelo"].map(lambda m: "GPU (MLX)" if "Parakeet" in m else "CPU")
comparativa["configuracion"] = comparativa["modelo"].map({k: v[1] for k, v in {**modelos_ocr, **modelos_asr}.items()})
comparativa = comparativa[["modalidad", "rol", "modelo", "metrica", "error", "mediana_por_caso",
                           "segundos", "seg_por_caso", "rtf", "equipo", "configuracion"]]

por_caso.to_csv(DATOS / "metricas_por_caso.csv", index=False)
comparativa.to_csv(DATOS / "comparativa_modelos.csv", index=False)
comparativa.drop(columns="configuracion").round(3)

**Decisión para un entorno real**

*(Se escribe después de ver los resultados: qué modelo se elegiría en cada modalidad considerando precisión, costo computacional y latencia.)*

**Reflexión por tipo de documento y por tipo de audio**

*(Se escribe después de ver los resultados: dónde hubo mejor y peor desempeño, y la causa.)*